In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("credit_card_fraud_cleaned.csv")

print(df.shape)
print(df.head())

(9994, 40)
  transaction_id  time_seconds  amount_inr        V1        V2        V3  \
0     TXN0008777       57630.0     1046.74  2.807697  0.276265  1.062996   
1     TXN0002200       38274.0      754.26 -1.975716 -0.490516 -0.795651   
2     TXN0009597       41803.0       50.00 -0.429719  1.043077  0.757801   
3     TXN0008491       58315.0       50.00  0.005155 -1.062797  0.572637   
4     TXN0003315       59128.0     3472.24 -0.328406  1.568590  1.523985   

         V4        V5        V6        V7  ...       V28  merchant_category  \
0 -0.221626 -0.577703 -0.210361 -1.309458  ... -0.919444     atm withdrawal   
1 -0.987246 -1.141452 -0.781381  0.709717  ...  0.839465      food & dining   
2 -0.214829 -0.737345  0.491581 -1.222717  ...  0.200888        electronics   
3 -0.993741 -0.778116  0.013816  1.951724  ...  1.074161          utilities   
4  1.196267 -0.721814 -1.233053  0.612499  ...  0.968516         healthcare   

    card_type   entry_mode  city_tier  is_foreign  custom

In [6]:
print(df.columns.tolist())

['transaction_id', 'time_seconds', 'amount_inr', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'merchant_category', 'card_type', 'entry_mode', 'city_tier', 'is_foreign', 'customer_age_group', 'account_age_months', 'hour_of_day', 'Class']


In [7]:
print(df["Class"].value_counts(dropna=False))

Class
0    9772
1     222
Name: count, dtype: int64


In [8]:
df = df.rename(columns={"Class": "is_fraud"})

In [10]:
# 1. Compare Fraud vs Non-Fraud Transactions
# ---------------------------------------------------------

fraud_counts = df["is_fraud"].value_counts()

print("Fraud vs Non-Fraud Transactions:")
print(fraud_counts)

Fraud vs Non-Fraud Transactions:
is_fraud
0    9772
1     222
Name: count, dtype: int64


In [11]:
# 2. Calculate Overall Fraud Rate
# ---------------------------------------------------------

fraud_rate = df["is_fraud"].mean() * 100

print(f"Overall Fraud Rate: {fraud_rate:.2f}%")

Overall Fraud Rate: 2.22%


In [12]:
# 3. Analyze Fraud by Merchant Category
# ---------------------------------------------------------

fraud_by_merchant = (
    df.groupby("merchant_category")["is_fraud"]
      .agg(
          total_transactions="count",
          fraud_transactions="sum",
          fraud_rate="mean"
      )
      .sort_values("fraud_rate", ascending=False)
)

fraud_by_merchant["fraud_rate"] *= 100

print("Fraud by Merchant Category:")
print(fraud_by_merchant)

Fraud by Merchant Category:
                   total_transactions  fraud_transactions  fraud_rate
merchant_category                                                    
atm                                 7                   1   14.285714
atm withdrawal                    971                  71    7.312049
online shopping                   956                  61    6.380753
electronics                      1708                  78    4.566745
grocery                           890                   3    0.337079
fuel                              918                   2    0.217865
retail                            922                   2    0.216920
utilities                         931                   2    0.214823
healthcare                        878                   1    0.113895
food & dining                     885                   1    0.112994
food                               15                   0    0.000000
travel                            913                   0    0

In [13]:
# 4. Analyze Fraud by Entry Mode
# ---------------------------------------------------------

fraud_by_entry_mode = (
    df.groupby("entry_mode")["is_fraud"]
      .agg(
          total_transactions="count",
          fraud_transactions="sum",
          fraud_rate="mean"
      )
      .sort_values("fraud_rate", ascending=False)
)

fraud_by_entry_mode["fraud_rate"] *= 100

print("Fraud by Entry Mode:")
print(fraud_by_entry_mode)

Fraud by Entry Mode:
             total_transactions  fraud_transactions  fraud_rate
entry_mode                                                     
online                     2933                 136    4.636891
swipe                      1894                  63    3.326294
chip                       3792                  20    0.527426
contactless                1375                   3    0.218182


In [16]:
# 5. Compare Transaction Amounts
#    Fraud vs Non-Fraud
# ---------------------------------------------------------

amount_comparison = (
    df.groupby("is_fraud")["amount_inr"]
      .agg(
          transaction_count="count",
          total_amount="sum",
          average_amount="mean",
          median_amount="median",
          maximum_amount="max"
      )
)

print("Transaction Amount Comparison:")
print(amount_comparison)

Transaction Amount Comparison:
          transaction_count  total_amount  average_amount  median_amount  \
is_fraud                                                                   
0                      9772   55415818.94     5670.877910        1046.74   
1                       222    4181956.90    18837.643694        1046.74   

          maximum_amount  
is_fraud                  
0             8171863.64  
1             3413796.72  


In [17]:
# 6. Analyze Foreign vs Domestic Transaction Fraud
# ---------------------------------------------------------

fraud_by_foreign = (
    df.groupby("is_foreign")["is_fraud"]
      .agg(
          total_transactions="count",
          fraud_transactions="sum",
          fraud_rate="mean"
      )
      .sort_values("fraud_rate", ascending=False)
)

fraud_by_foreign["fraud_rate"] *= 100

print("Foreign vs Domestic Fraud:")
print(fraud_by_foreign)

Foreign vs Domestic Fraud:
            total_transactions  fraud_transactions  fraud_rate
is_foreign                                                    
TRUE                       845                  89   10.532544
FALSE                     9132                 133    1.456417
1                            6                   0    0.000000
0                           11                   0    0.000000


In [19]:
# 7. Identify High-Value Fraudulent Transactions
# ---------------------------------------------------------

fraud_transactions = df[df["is_fraud"] == 1].copy()

high_value_fraud = (
    fraud_transactions
    .sort_values("amount_inr", ascending=False)
    .head(20)
)

print("Top 20 High-Value Fraudulent Transactions:")
print(high_value_fraud[
    ["transaction_id", "amount_inr", "merchant_category",
     "entry_mode", "is_foreign", "is_fraud"]
])

Top 20 High-Value Fraudulent Transactions:
     transaction_id  amount_inr merchant_category entry_mode is_foreign  \
402      TXN0000971  3413796.72   online shopping     online       TRUE   
668      TXN0007943    11976.82    atm withdrawal     online      FALSE   
4033     TXN0008179    11933.06    atm withdrawal     online      FALSE   
74       TXN0000994    11903.11       electronics      swipe       TRUE   
6101     TXN0009150    11871.17    atm withdrawal     online       TRUE   
9577     TXN0009754    11843.49       electronics     online       TRUE   
583      TXN0009921    11812.36       electronics       chip      FALSE   
1393     TXN0004522    11757.92    atm withdrawal     online      FALSE   
90       TXN0004685    11726.18   online shopping     online       TRUE   
731      TXN0002751    11704.57   online shopping      swipe       TRUE   
3168     TXN0003123    11654.21   online shopping     online       TRUE   
6929     TXN0008314    11645.79    atm withdrawal     onl

In [21]:
# 8. Examine Transaction-Level Fraud Patterns
# ---------------------------------------------------------

fraud_transaction_summary = fraud_transactions[
    ["transaction_id", "amount_inr", "merchant_category",
     "entry_mode", "is_foreign"]
].sort_values("amount_inr", ascending=False)

print("Fraudulent Transaction-Level Patterns:")
print(fraud_transaction_summary.head(20))

Fraudulent Transaction-Level Patterns:
     transaction_id  amount_inr merchant_category entry_mode is_foreign
402      TXN0000971  3413796.72   online shopping     online       TRUE
668      TXN0007943    11976.82    atm withdrawal     online      FALSE
4033     TXN0008179    11933.06    atm withdrawal     online      FALSE
74       TXN0000994    11903.11       electronics      swipe       TRUE
6101     TXN0009150    11871.17    atm withdrawal     online       TRUE
9577     TXN0009754    11843.49       electronics     online       TRUE
583      TXN0009921    11812.36       electronics       chip      FALSE
1393     TXN0004522    11757.92    atm withdrawal     online      FALSE
90       TXN0004685    11726.18   online shopping     online       TRUE
731      TXN0002751    11704.57   online shopping      swipe       TRUE
3168     TXN0003123    11654.21   online shopping     online       TRUE
6929     TXN0008314    11645.79    atm withdrawal     online      FALSE
5113     TXN0009418    11

In [22]:
# 9. Compare Fraud Across Customer / Transaction
#    Characteristics
# ---------------------------------------------------------

# Example: fraud by customer characteristics
# Replace these columns with the actual columns in your dataset.

customer_characteristics = [
    "customer_age",
    "customer_gender"
]

for column in customer_characteristics:

    if column in df.columns:

        result = (
            df.groupby(column)["is_fraud"]
              .agg(
                  total_transactions="count",
                  fraud_transactions="sum",
                  fraud_rate="mean"
              )
              .sort_values("fraud_rate", ascending=False)
        )

        result["fraud_rate"] *= 100

        print(f"\nFraud by {column}:")
        print(result)

In [23]:
# 10. Fraud by Multiple Transaction Characteristics
# ---------------------------------------------------------

fraud_combination = (
    df.groupby(
        ["merchant_category", "entry_mode", "is_foreign"]
    )["is_fraud"]
    .agg(
        total_transactions="count",
        fraud_transactions="sum",
        fraud_rate="mean"
    )
    .sort_values("fraud_rate", ascending=False)
)

fraud_combination["fraud_rate"] *= 100

print("Fraud by Combined Transaction Characteristics:")
print(fraud_combination.head(20))

Fraud by Combined Transaction Characteristics:
                                          total_transactions  \
merchant_category entry_mode  is_foreign                       
atm               online      FALSE                        1   
atm withdrawal    online      TRUE                        34   
online shopping   swipe       TRUE                        17   
                  online      TRUE                        51   
atm withdrawal    swipe       TRUE                        21   
electronics       swipe       TRUE                        31   
                  online      TRUE                        67   
atm withdrawal    online      FALSE                      299   
online shopping   online      FALSE                      246   
electronics       swipe       FALSE                      281   
atm withdrawal    swipe       FALSE                      147   
electronics       online      FALSE                      455   
atm withdrawal    chip        TRUE                       

In [24]:
df.to_csv("credit_card_fraudulent transaction behavior.csv", index=False)